In [1]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings

/Users/rachin/Desktop/GenAI/genai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = TextLoader("../speech.txt")
data = loader.load()
data

[Document(metadata={'source': '../speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 30)
splitter = text_splitter.split_documents(data)

In [4]:
embedding = OllamaEmbeddings(model = "gemma:2b")

/var/folders/41/r75gksp11wl_rrwbrwywkbjh0000gn/T/ipykernel_3079/2966033954.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding = OllamaEmbeddings(model = "gemma:2b")


In [5]:
vectordb = Chroma.from_documents(splitter, embedding)
vectordb

In [6]:
## querying
query = "What does the speaker believe is the main reason the United States should enter the war?"

docs = vectordb.similarity_search(query)
docs[0].page_content

'our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

In [7]:
#saving to the disk

vectordb = Chroma.from_documents(splitter, embedding, persist_directory = "./chroma_db")

In [8]:
#load from disk

db2 = Chroma(persist_directory="./chroma_db",embedding_function=embedding)

docs= db2.similarity_search(query)
docs[0].page_content

'our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

In [9]:
docs = vectordb.similarity_search_with_score(query)
docs

[(Document(id='c408933c-3c00-4cd8-8685-01eab248a19a', metadata={'source': '../speech.txt'}, page_content='our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  3510.24365234375),
 (Document(id='bd71cc5a-f596-4641-8745-27c88298e45b', metadata={'source': '../speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of

In [10]:
retriever = vectordb.as_retriever()

retriever.invoke(query)[0].page_content

'our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'